# 제2장 제2절 pandas 데이터 처리

2027 개정 원고의 실습 자료입니다. 위에서 아래로 실행합니다. 데이터 파일 준비와 버전은 [실행 안내](https://github.com/Lilcob/AICE_Associate/blob/main/docs/getting_started.md)를 확인하세요.

건강검진 CSV와 iris.data를 노트북의 현재 폴더에 준비합니다. 실제 건강검진 자료는 2023년 자료이며 개정판 연도와 구분합니다.

설치나 버전 변경이 필요하면 안내에 따라 준비한 뒤 런타임을 재시작하세요. 저장된 출력은 비워 두었으며 직접 실행한 값을 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
print('NumPy', np.__version__, '| pandas', pd.__version__)


# 핵심 개념: 데이터 분석의 스위스 아미 나이프

## Pandas Series & DataFrame

In [ ]:
import pandas as pd                                      # ①
series = pd.Series([10, 20, 30], index=['a', 'b', 'c'])  # ②
print(series)                                            # ③
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})      # ④
print(df)                                                # ⑤

## 데이터 불러오고 확인하기




In [ ]:
import pandas as pd
# 전체 데이터 불러오기
df = pd.read_csv("국민건강보험공단_건강검진정보_2023.CSV", encoding="cp949")  # ①
print(df.head())  # ②
print(df.columns)  # ③
print(df.describe()) # ④
# 특정 열만 선택하여 불러오기
df_selected = pd.read_csv("국민건강보험공단_건강검진정보_2023.CSV",
                usecols=["기준년도", "성별코드"], encoding="cp949") # ⑤
print(df_selected.head()) # ⑥
print(df_selected.columns) # ⑦

In [ ]:
import requests, json
import pandas as pd
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url, timeout=20)
response.raise_for_status()
users_data = response.json()
df_users = pd.DataFrame(users_data) # ①
print(df_users.head()) # ②
with open("users.json", "w", encoding="utf-8") as f: # 파일 저장
   json.dump(users_data, f, ensure_ascii=False, indent=2)
df_local = pd.read_json("users.json") # ③
print(df_local.head()) # ④

## 데이터 필터링

###Label/Integer-location 기반 인덱싱(loc, iloc)


In [ ]:
sample_df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['x', 'y', 'z']) # ①
print(sample_df.loc['x', 'A']) # ②
print(sample_df.iloc[0, 1]) # ③

### 컬럼명, 인덱싱으로 데이터 선택

In [ ]:
column_df = df[['기준년도', '성별코드']] # ①
print(column_df.head()) # ②
label_indexing_df = df.loc[0:4, ['기준년도', '성별코드', '가입자일련번호', '시력(좌)', '시력(우)']] # ③
print(label_indexing_df.head())  # ④
integer_indexing_df = label_indexing_df.iloc[0:2, 0:4] # ⑤
print(integer_indexing_df.head()) # ⑥

### 불리언 인덱싱(Boolean Indexing)

In [ ]:
male_df = df[df['성별코드'] == 1] # ①
print(male_df.head()['성별코드']) # ②
eye_sight_df = df[(df['시력(좌)'] > 1.0) & (df['시력(우)'] > 0.5)] # ③
print(eye_sight_df.head().loc[:, ['시력(좌)', '시력(우)']]) # ④

###query 메서드로 복잡한 조건 검색

In [ ]:
threshold = 0.5 # ①
result = df.query("흡연상태 >= 1.0") # ②
result = result.query("`시력(좌)` >= @threshold or `시력(우)` >= @threshold") # ③
print(result.head().loc[:, ['흡연상태', '시력(좌)', '시력(우)']]) # ④

## 데이터 추가, 삭제, 복사, 변경 (+inplace 및 체이닝)

### 데이터 추가, 변경, 복사

In [ ]:
import pandas as pd
df = pd.DataFrame({
   'A': [1, 2, 3],
   'B': [4, 5, 6]
})
df = df.assign(C=df['A'] + df['B'])   # ①
print(df)
df['B'] = df['B'].replace({5: 50})    # ②
print(df)
df['A'] = df['A'] * 2                 # ③
print(df)
df.insert(1, 'D', [100, 200, 300])    # ④
print(df)
df_deep = df.copy()                   # ⑤
df_deep.loc[0, 'A'] = 100             # ⑥
print(df_deep)
print(df)
df_alias = df                       # ⑦
df_alias.loc[0, 'A'] = 100          # ⑧
print(df)

In [ ]:
df = pd.DataFrame({
  'A': [1, 2, 3],
  'B': [4, 5, 6]
})
result = (df
         .assign(C=df['A'] + df['B'])     # ①
         .replace({'B': {5: 50}})       # ②
         .assign(A=lambda x: x['A'] * 2)  # ③
)
result.insert(1, 'D', [100, 200, 300])    # ④
print(result)

###컬럼/행 삭제, 중복 제거, 일부만 추출

In [ ]:
import pandas as pd
df = pd.DataFrame({
   "A": [1, 2, 2, 4],
   "B": [5, 6, 6, 8],
   "C": [9, 10, 10, 12],
   "D": [13, 14, 14, 16]
})
print(df.drop('D', axis=1))                        # ①
print(df.drop(1, axis=0))                          # ②
print(df.drop_duplicates())                        # ③
print(df.drop_duplicates(subset=["A", "B"]))       # ④
print(df.truncate(before=1, after=2))              # ⑤
print(df.truncate(before=1, after=3)[["A", "C"]])  # ⑥

## 결측치 처리 (fillna, dropna)

In [ ]:
import pandas as pd
df = pd.DataFrame({'A': [1, None, 3], 'B': [4, 5, None]})
print(df.fillna(0))                        # ①
print(df.dropna())                         # ②
print(df.dropna(axis=0, how='all'))        # ③
print(df.dropna(subset=['A']))             # ④

In [ ]:
import pandas as pd
df = pd.DataFrame({
   'A': [1, None, None, 4],
   'B': [None, 2, None, 5]
})
print(df.ffill())   # ①
print(df.bfill())   # ②

## 데이터 타입 변환

In [ ]:
import pandas as pd
df = pd.DataFrame({'A': ['1', '2', '3'], 'B': [4, 5, 6]})
df['A'] = df['A'].astype(int) # ①
print(df.dtypes)
df = pd.DataFrame({'date': ['2023-01-01','2023-02-01','2023-03-01']})
df['date'] = pd.to_datetime(df['date']) # ②
print(df.dtypes)
df = pd.DataFrame({'value': ['10', '20', '30', 'error']})
df['value_num'] = pd.to_numeric(df['value'], errors='coerce') # ③
print(df)
df = pd.DataFrame({'grade': ['A', 'B', 'A', 'C', 'B']})
df['grade'] = df['grade'].astype('category') # ④
print(df.dtypes)  
df = pd.DataFrame({'X': [1, 2, 3], 'Y': ['a', 'b', 'c']})
df = df.infer_objects() # ⑤
print(df.dtypes)  

## 그룹화(groupby)와 집계

In [ ]:
import pandas as pd  # 
df = pd.DataFrame({
   '성별코드': [1, 1, 2, 2, 1, 2],  # ①
   '학년': ['1학년', '2학년', '1학년', '2학년', '1학년', '2학년'],  # ②
   '체중(kg)': [60, 65, 55, 58, 70, 62],  # ③
   '키(cm)': [165, 168, 160, 162, 170, 159]  # ④
})
grouped_single = df.groupby('성별코드').agg({
   '체중(kg)': 'mean',
   '키(cm)': ['max', 'std']
})  # ⑤
print(grouped_single)
grouped_multi = df.groupby(['성별코드', '학년']).agg({
   '체중(kg)': ['mean', 'max'],
   '키(cm)': ['min', 'std']
})  # ⑥
print(grouped_multi)

범주형의 메모리·실행 시간 비교는 [확장 실습](https://github.com/Lilcob/AICE_Associate/blob/main/docs/extended_notes.md#category-benchmark)으로 옮겼습니다.

## 데이터 병합과 추가 - Join(merge), Union(concat)

###Join(merge)

In [ ]:
import pandas as pd
df = pd.DataFrame({
'가입자일련번호': [34735, 4105118, 362482, 653166, 4152237, 3587054, 4045005],
'체중(5kg단위)': [70, 55, 65, 70, 65, 60, 75],
'허리둘레': [80.0, 90.0, 96.0, 85.0, 84.5, 69.2, 80.5]
})
df1 = df[["가입자일련번호", "체중(5kg단위)"]].head(5) # ①
df2 = df[["가입자일련번호", "허리둘레"]].iloc[2:7]
print(df1)
print(df2)
inner_merged = pd.merge(df1, df2, on="가입자일련번호", how="inner") # ②
outer_merged = pd.merge(df1, df2, on="가입자일련번호", how="outer")
left_merged = pd.merge(df1, df2, on="가입자일련번호", how="left")
right_merged = pd.merge(df1, df2, on="가입자일련번호", how="right")
print(inner_merged)
print(outer_merged)
print(left_merged)
print(right_merged) 

###Concat

In [ ]:
concat_row = pd.concat([df1.head(1), df2.tail(1)], axis=0)  # ①
print(concat_row)

In [ ]:
df_part1 = pd.DataFrame({ "가입자일련번호": [100, 101, 102],"체중": [70, 65, 80]})# ①
df_part2 = pd.DataFrame({"가입자일련번호": [101, 102, 103],"허리둘레": [85, 90, 95]})
merged_compare = pd.merge(df_part1,
   df_part2,on="가입자일련번호",how="outer")  # ②
concat_compare = pd.concat([df_part1, df_part2]).drop_duplicates()  # ③
print(merged_compare)
print(concat_compare)

## 정렬


In [ ]:
import pandas as pd
df = pd.DataFrame({
  '가입자일련번호': [34735, 4105118, 362482, 653166, 4152237, 5224137, 2362482, 1653166],
  '체중(5kg단위)': [70, 55, 65, 52, 54, 68, 70, 65],
  '허리둘레': [80.0, 90.0, 90.0, 90.0, 84.5, 86.2, 81.5, 78.7]
})
copied_head = df.head().copy()  # ①
print(copied_head.loc[:, ['허리둘레']])  # ②
copied_head.sort_values(by='허리둘레', ascending=True, kind='stable', inplace=True)  # ③
print(copied_head.loc[:, ['허리둘레']])  # ④

In [ ]:
sorted_df = df.sort_values(by=['허리둘레', '체중(5kg단위)'], ascending=[False, True])  # ⑤
print(sorted_df.loc[:, ['허리둘레', '체중(5kg단위)']].head())  # ⑥
df_sorted_index = df.sort_index()  # ⑦
df_sorted_columns = df.sort_index(axis=1, ascending=False)  # ⑧
print(df_sorted_index.head())  # ⑨
print(df_sorted_columns.head())  # ⑩

## 인덱스 재설정(reset_index)

In [ ]:
import pandas as pd
df = pd.DataFrame({
   '가입자일련번호': [34735, 4105118, 362482, 653166, 4152237],
   '체중(5kg단위)': [70, 55, 65, 70, 65],
   '허리둘레': [80.0, 90.0, 96.0, 85.0, 84.5]
})
sorted_df = df.sort_values(by='허리둘레', ascending=True)  # ①
print(sorted_df)
reset_df = sorted_df.reset_index()  # ②
print(reset_df)
reset_drop_df = sorted_df.reset_index(drop=True)  # ③
print(reset_drop_df)
sorted_df_copy = sorted_df.copy()  # ④
sorted_df_copy.reset_index(drop=True, inplace=True)  # ⑤
print(sorted_df_copy)

### DataFrame의 map과 apply

In [ ]:
import pandas as pd
df = pd.DataFrame({
   '가입자일련번호': [34735, 4105118, 362482, 653166, 4152237],
   '체중(5kg단위)': [70, 55, 65, 70, 65],
   '허리둘레': [80.0, 90.0, 96.0, 85.0, 84.5]
})
weight_map = df['체중(5kg단위)'].map(lambda x: '중량' if x >= 70 else '경량')  # ①
print(weight_map)
df['체중 카테고리'] = weight_map  # ②
print(df)
def sum_row(row):  # ③
 return row['체중(5kg단위)'] + row['허리둘레']
df['합계'] = df.apply(sum_row, axis=1)  # ④
print(df)
mean_col = df[['체중(5kg단위)', '허리둘레']].apply(lambda col: col.mean(), axis=0)  # ⑤
print(mean_col.round(2))
multi_apply = df[['체중(5kg단위)', '허리둘레']].apply([lambda col: col.mean(), lambda col: col.std()])  # ⑥
print(multi_apply.round(2))

### 기초 통계량 분석(평균, 중앙값, 최빈값, 최댓값, 최솟값, 표준편차, 분산)

In [ ]:
import pandas as pd
df = pd.DataFrame({
   '가입자일련번호': [34735, 4105118, 362482, 653166, 4152237],
   '체중(5kg단위)': [70, 55, 65, 70, 65],
   '허리둘레': [80.0, 90.0, 96.0, 85.0, 84.5]
})
desc = df[['허리둘레', '체중(5kg단위)']].describe()  # ①
print(desc.round(2))
mean_waist = df['허리둘레'].mean()  # ②
print(round(mean_waist, 2))
median_waist = df['허리둘레'].median()  # ③
print(round(median_waist, 2))
mode_weight = df['체중(5kg단위)'].mode()  # ④
print(mode_weight)
max_waist = df['허리둘레'].max()  # ⑤
print(max_waist)
min_waist = df['허리둘레'].min()  # ⑥
print(min_waist)
std_waist = df['허리둘레'].std()  # ⑦
print(round(std_waist, 2))
var_weight = df['체중(5kg단위)'].var()  # ⑧
print(round(var_weight, 2))

# 단계별 실습: 붓꽃 품종 분류 실습


##붓꽃(Iris) 품종 분류를 통한 식물 자동 인식 시스템

Iris 데이터셋 컬럼 설명(데이터 파일명(csv): iris.data)
- sepal_length (꽃받침 길이, cm) : 꽃받침(Sepal)의 길이를 센티미터 단위로 나타낸 값입니다.
- sepal_width (꽃받침 너비, cm) : 꽃받침(Sepal)의 폭(넓이)을 센티미터 단위로 나타낸 값입니다.
- petal_length (꽃잎 길이, cm) : 꽃잎(Petal)의 길이를 센티미터 단위로 나타낸 값입니다.
- petal_width (꽃잎 너비, cm) : 꽃잎(Petal)의 폭을 센티미터 단위로 나타낸 값입니다.
- species (품종 / 종 이름) : 붓꽃(Iris) 식물의 세 가지 종(setosa, versicolor, virginica)을 나타냅니다.


###데이터 불러오고 확인하기


연습 문제 1: pandas 와 numpy 라이브러리를 불러오고 주어진 데이터셋을 DataFrame 으로 읽어 변수 df 에 할당하는 코드를 작성하세요.
1. 데이터 파일명 : iris.data (파일은 본 문제 와 동일한 경로에 있습니다)
2. 컬럼명은 순서대로 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'로 설정하세요.

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("iris.data", 
names=['sepal_length', 'sepal_width', 'petal_length',
       'petal_width', 'species'
]) # ①

연습 예시 2: iris DataFrame의 기본 정보를 확인하세요.
1. 데이터의 shape 을 df_shape 변수에 할당하세요
2. 각 컬럼의 데이터 타입을 확인하여 차례대로 df_type 변수에 할당하세요
3. 결측값 개수를 df_missing_values 변수에 할당하세요
4. 첫 5행을 df_head 변수에 할당하세요


In [ ]:
df_shape = df.shape  # ①
df_type = df.dtypes  # ②
df_missing_values = df.isnull().sum()  # ③
df_head = df.head()  # ④
print(df_shape)
print(df_type)
print(df_missing_values)
print(df_head)

연습 문제3: 데이터의 구조를 미리 확인하기 위해 iris DataFrame의 처음 10개 행과 마지막 5개 행을 출력하세요.
1. 각각의 결과를 df_first10, df_last5 변수에 할당하세요.


In [ ]:
df_first10 = df.head(10)  # ①
df_last5 = df.tail(5)  # ②
print(df_first10)
print(df_last5)

연습 문제4: iris DataFrame에서 수치형 컬럼들의 기술 통계량을 확인하세요(평균, 표준편차, 최솟값, 최댓값, 사분위수 포함)
1. 결과를 df_stats 변수에 할당하세요

In [ ]:
df_stats = df[['sepal_length', 'sepal_width', 'petal_length',
 'petal_width']].describe()  # ①
print(df_stats)

연습 문제5: iris DataFrame에서 species 컬럼의 고유값(품종 종류)과 각 값이 나타나는 개수를 구하세요.
1. 결과를 df_species_counts 변수에 할당하세요.

In [ ]:
df_species_counts = df['species'].value_counts()  # ①
print(df_species_counts)

연습 문제6: iris DataFrame의 각 수치형 컬럼(sepal_length, sepal_width, petal_length, petal_width)의 평균(mean)을 계산하세요.
1. 결과를 df_mean 변수에 저장하세요.

In [ ]:
df_mean = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].mean()  # ①
print(df_mean)

연습 문제7: iris DataFrame을 iris.json 파일로 저장하고, 다시 불러와 변수에 저장하세요.
1. 저장 시 orient='records' 옵션을 사용하세요.
2. 결과를 df_json 변수에 할당하세요.


In [ ]:
df.to_json('iris.json', orient='records')  # ①
df_json = pd.read_json('iris.json')  # ②
print(df_json.head(3))

연습 문제8: iris DataFrame을 iris.xlsx 파일로 저장하고, 다시 불러와 변수에 저장하세요.
1. 결과를 df_xlsx 변수에 할당하세요.


In [ ]:
df.to_excel('iris.xlsx', index=False)  # ①
df_xlsx = pd.read_excel('iris.xlsx')  # ②
print(df_xlsx.head(3))

###데이터 필터링

연습 문제 9: species가 'Iris-setosa' 인 행만 필터링하여 변수에 저장하세요.
1. 결과를 df_setosa 변수에 할당하세요.


In [ ]:
df_setosa = df[df['species'] == 'Iris-setosa']  # ①
print(df_setosa.tail(3))

연습 문제 10: 꽃잎 길이(petal_length)가 5 이상인 행만 선택하여 변수에 저장하세요.
1. 결과를  df_petal5 변수에 할당하세요

In [ ]:
df_petal5 = df[df['petal_length'] >= 5]  # ①
print(df_petal5.head(3))

연습 문제 11: 꽃받침 길이(sepal_length)가 6 이상이고, 꽃잎 너비(petal_width)가 2 이하인 행만 선택하세요.
1. 결과를 df_condition 변수에 저장하세요.

In [ ]:
df_condition = df[(df['sepal_length']>=6) & (df['petal_width']<=2)]  # ①
print(df_condition.head(3))

연습 문제 12: iloc를 사용하여 iris DataFrame의 첫 3행과 첫 2열을 선택하세요.
1. 결과를 df_iloc 변수에 저장하세요.


In [ ]:
df_iloc = df.iloc[:3, :2]  # ①
print(df_iloc)

연습 문제 13: query 메서드를 이용해 species == "virginica" 이면서 sepal_width > 3인 데이터를 추출하세요.
1. df_query 변수에 저장하세요.

In [ ]:
df_query = df.query("species == 'Iris-virginica' and sepal_width > 3") # ①
print(df_query)

###데이터 추가, 삭제, 복사, 변경


연습 문제 14: df의 얕은 복사와 깊은 복사를 각각 df_shallow, df_deep에 저장하세요. df.copy(deep=False)와 df.copy()를 사용합니다. df_alias = df는 복사와 구분합니다.

In [ ]:
df_shallow = df.copy(deep=False)  # 얕은 복사
df_deep = df.copy()  # ②

연습 문제 15: 위 결과에서 원본을 수정하지 않는 변수를 선택하여 새로운 컬럼 sepal_area (꽃받침 길이 × 꽃받침 너비)를 추가하여 df_working 변수에 할당하세요.

In [ ]:
df_deep['sepal_area'] = df_deep['sepal_length']*df_deep['sepal_width'] # ①
df_working = df_deep
print(df_working['sepal_area'].head(3))

연습 문제 16: df_working 변수에 할당된 DataFrame 의 sepal_width 컬럼을 삭제하세요.
1. 결과를 df_drop 변수에 저장하세요.

In [ ]:
df_drop = df_working.drop(columns='sepal_width')  # ①
print(df_drop.head(3))

연습 문제 17: df_drop 변수에 할당된 DataFrame 의 species 컬럼에서 'Iris-setosa'를 'Setosa'로, 'Iris-virginica' 를 'Virginica' 로, 'Iris-versicolor' 를 'Versicolor' 로 변경하세요.
1. 결과를 df_replace 변수에 저장하세요.


In [ ]:
df_replace = df_drop.replace({
   'species': {
       'Iris-setosa': 'Setosa',
       'Iris-virginica': 'Virginica',
       'Iris-versicolor': 'Versicolor'
   }
})
print(df_replace['species'].value_counts())

###결측치 처리

연습 문제 18: df_replace 데이터프레임에 아래의 새 관측값 new_data 를 추가하여 변수에 저장하세요. 단, 일부 행에는 결측치(NaN)가 포함되어 있습니다.
1. 각 데이터프레임을 추가할 때 기존 인덱스를 무시하고 새로운 인덱스를 가지도록 하여 df_update 변수에 할당하세요
2. 전체 컬럼에서 결측치인 행을 모두 제거한 결과를 df_dropna_full 변수에 저장하세요.
3. 특정 컬럼(sepal_length, petal_width)이 모두 결측치인 행만 제거한 결과를 df_dropna_partial 변수에 저장하세요.


In [ ]:
new_data = pd.DataFrame({
    'sepal_length': [5.2, np.nan, 6.0],
    'petal_length': [1.5, 4.8, 5.5],
    'petal_width': [0.3, np.nan, np.nan],
    'species': ['Setosa', 'Versicolor', 'Virginica']
})

In [ ]:
df_updated = pd.concat([df_replace, new_data], ignore_index=True) # ①
df_dropna_full = df_updated.dropna(how='any') # ②
df_dropna_partial = df_updated.dropna(
        subset=['sepal_length', 'petal_width'], 
        how='all'
      )  # ③
print(df_updated['species'].unique())
print(df_updated.tail(3))
print(df_dropna_full.tail(3))
print(df_dropna_partial.tail(3))

연습 문제 19: df_dropna_partial 변수에 할당된 DataFrame 의 결측치를 각 컬럼의 평균값으로 채운 결과를 df_fillna 변수에 할당하고. df_dropna_partial 변수와 df_fillna 변수의 컬럼별 결측치를 prev_nan_count 와 post_nan_count 변수에 각각 저장하세요

In [ ]:
df_fillna = df_dropna_partial.fillna(
                               df_dropna_partial.mean(numeric_only=True)
                )
prev_nan_count = df_dropna_partial.isnull().sum()
print(prev_nan_count)
post_nan_count = df_fillna.isnull().sum()
print(post_nan_count)

###데이터 타입 변환


연습 문제 20: df_fillna 변수에 할당 된 DataFrame 의 species 컬럼을 범주형(category) 타입으로 변환하고 df_cat 변수에 저장하세요.

In [ ]:
df_cat = df_fillna.astype({'species': 'category'})
print(df_cat.dtypes)

###그룹화와 집계

연습 문제 21: df_cat 변수에 할당된 DataFrame 에서 species별로 sepal_length 의 평균을 구해 df_group_mean 변수에 저장하세요.

In [ ]:
df_group_mean = df_cat.groupby('species', observed=True)['sepal_length'].mean()
print(df_group_mean)

연습 문제 22: species별 꽃잎 길이(petal_length)의 최댓값과 최솟값을 구해 df_group_stats 변수에 저장하세요.

In [ ]:
df_group_stats = df_cat.groupby('species', observed=True)['petal_length'].agg(['max', 'min'])
print(df_group_stats)

###데이터 병합과 추가

연습 문제 23: df_cat 변수에 할당된 DataFrame 을 꽃받침(sepal) 관련과 꽃잎(petal) 관련 열로 분리하고 pd.concat을 사용해 열 방향으로 합치세요.
1. df_concat 변수에 저장하세요.

In [ ]:
sepal_df = df_cat[['sepal_length', 'sepal_area']]
petal_df = df_cat[['petal_length', 'petal_width']]
df_concat = pd.concat([sepal_df, petal_df], axis=1)
print(df_concat.head(3))

연습 문제 24: 원래 행 인덱스를 보존한 df_cat에 df의 sepal_width를 인덱스 기준 left 병합하세요. validate="one_to_one"을 지정해 행이 불어나지 않는지 확인하고 df_merged에 저장합니다.

In [ ]:
df_merged = pd.merge(
    df_cat, df[['sepal_width']],
    left_index=True, right_index=True,
    how='left', validate='one_to_one',
)
assert len(df_merged) == len(df_cat)
print(df_merged.shape)
common_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
diff_df = df_merged.loc[~df_merged.index.isin(df.index), common_cols]
print(diff_df)

###정렬

연습 문제 25: df_merged 변수에 할당된 DataFrame 에서 꽃잎 길이(petal_length) 기준으로 내림차순 정렬하세요.
1. 결과를 df_sorted 변수에 저장하세요.

In [ ]:
df_sorted = df_merged.sort_values(by='petal_length', ascending=False, kind='stable')
print(df_sorted.head(3))

###인덱스 재설정


연습 문제 26: df_sorted 변수에 할당된 DataFrame 에서 인덱스를 재설정(reset_index) 하세요.
1. df_reset 변수에 저장하며 기존 인덱스는 삭제하세요.

In [ ]:
df_reset = df_sorted.reset_index(drop=True)
print(df_reset.tail(3))

###map과 apply

연습 문제 27: df_reset 변수에 할당된 DataFrame 에 꽃받침 길이(sepal_length)가 5 이상이면 "Long", 아니면 "Short"라는 값을 가지는 새로운 컬럼 sepal_length_type을 추가하세요.

In [ ]:
df_reset['sepal_length_type'] = (df_reset['sepal_length'] >= 5).map({True: 'Long', False: 'Short'})
print(df_reset['sepal_length_type'].value_counts())

연습 문제 28: df_reset 변수에 할당된 DataFrame 에 꽃잎 길이(petal_length)와 꽃잎 너비(petal_width)의 합을 계산하는 사용자 정의 함수(calc_petal_sum)를 만들고 apply를 이용해 새로운 컬럼 petal_sum을 추가하세요.

In [ ]:
def calc_petal_sum(row):
   return row['petal_length'] + row['petal_width']
df_reset['petal_sum'] = df_reset.apply(calc_petal_sum, axis=1)
print(df_reset[['petal_length', 'petal_width', 'petal_sum']].head())

연습 문제 29: df_reset 변수에 할당된 DataFrame 에서 apply 함수를 사용하여 각 행에 대해 'sepal_length + petal_length'와 'sepal_width + petal_width' 중 큰 값을 계산하는 함수(max_sum)를 작성하고'max_sum' 컬럼에 저장하세요.

In [ ]:
def max_sum(row):
   val1 = row['sepal_length'] + row['petal_length']
   val2 = row['sepal_width'] + row['petal_width']
   return max(val1, val2)
df_reset['max_sum'] = df_reset.apply(max_sum, axis=1)
print(df_reset[['sepal_length', 'sepal_width',
  'petal_length', 'petal_width', 'max_sum']].head())

###기초 통계량 분석

연습 문제 30: df_reset 변수에 할당된 DataFrame 에서 각 수치형 컬럼의 중앙값(median)을 구하세요.
1. 결과를 df_median 변수에 저장하세요.

In [ ]:
df_median = df_reset.median(numeric_only=True)
print(df_median)

연습 문제 31: df_reset 변수에 할당된 DataFrame 에서 각 수치형 컬럼의 표준편차(std)를 구하세요.
1. 결과를 df_std 변수에 저장하세요.

In [ ]:
df_std = df_reset.std(numeric_only=True)
print(df_std)

연습 문제 32: 다음 작업을 순서대로 수행하는 함수를 작성하세요.
1. df_reset 변수에 할당된 DataFrame에 JSON 형태로 주어진 region 정보를 추가하세요.
2. species 컬럼을 기준으로 region 컬럼을 생성하며, 기존 DataFrame을 직접 수정하도록 inplace를 활용하세요.
3. region 별로 다음 수치형 컬럼의 평균(mean) 과 표준편차(std) 를 계산하세요.  
(sepal_length, sepal_width, petal_length, petal_width, sepal_area)
4. 계산한 통계 결과를 CSV 파일로 저장하세요.  
(파일명: "iris_summary.csv")

함수는 최종 통계 DataFrame을 반환해야 합니다. json_data 는 다음과 같이 주어집니다.  

json_data = StringIO('{"Setosa": "Asia", "Versicolor": "Europe", "Virginica": "America"}')

In [ ]:
import json
json_data = '{"Setosa": "Asia", "Versicolor": "Europe", "Virginica": "America"}'
def process_iris_data(frame, mapping_json):
    missing_counts = frame.isna().sum()
    print('결측값이 있는 컬럼:')
    print(missing_counts[missing_counts > 0])
    frame['sepal_width'] = frame['sepal_width'].fillna(frame['sepal_width'].median())
    region_map = json.loads(mapping_json)
    frame['region'] = frame['species'].map(region_map)
    columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'sepal_area']
    stats = frame.groupby('region', observed=True, sort=True)[columns].agg(['mean', 'std'])
    stats.to_csv('iris_summary.csv')
    return stats
result = process_iris_data(df_reset, json_data)
print(result)

## 개념도와 추가 설명

[핵심 개념 노트북](https://colab.research.google.com/github/Lilcob/AICE_Associate/blob/main/notebooks/2027_concepts.ipynb)에서 개념도의 작은 예시를 실행할 수 있습니다.
